# Step 10: does the model use its rule when the moves are written out? (added after looking)

Plan and expectations committed before these answers existed: see [PLAN_more_inputs.md](../PLAN_more_inputs.md).

Test 3 again (`gemma-3-4b-it`, the same 1,296 days, 2017–2019), with two lines added below the table: the direction of the 1-, 5- and 20-day moves, and Kim's 12 indicators on day −1. Three runs: planted tester, planted none (its twin), real none.

`run_with_indicators.py` collects the answers. This notebook only reads them.

In [ ]:
import sys
sys.path.insert(0, "..")  # the main project's code is one folder up

import json

import pandas as pd
from sklearn.metrics import roc_auc_score

from evaluation import mcnemar
from llm_harness import rules
from llm_harness.client import SAVED
from llm_harness.questions import days_before
from llm_harness.run_feedback import load

RUNS = [("planted", "tester"), ("planted", "none"), ("real", "none")]

def answers(dataset, kind, suffix):
    """One row per answer, with its month number (1 = the first month)."""
    months = json.loads((SAVED / f"feedback_{dataset}_{kind}{suffix}_state.json").read_text(encoding="utf-8"))["months"]
    frame = pd.DataFrame([dict(r, month=n + 1) for n, month in enumerate(months) for r in month["results"]])
    frame["right"] = (frame["answer"] == "up") == frame["up"]
    return frame.set_index(["market", "day"])

new = {(d, k): answers(d, k, "_indicators") for d, k in RUNS}
old = {(d, k): answers(d, k, "") for d, k in RUNS}  # test 3
print({key: len(a) for key, a in new.items()})

## Was reading the table the problem?

Months 25–36: how often the answer matches what its own named rule gives. Test 3 got 47–55%. Answers with "none of these" are left out.

In [ ]:
def late_check(a, dataset):
    markets = load(dataset)
    late = a[a["month"] > 24].reset_index()
    late["rule's answer"] = [rules.implied_answer(days_before(markets[m], pd.Timestamp(d)), rule)
                             for m, d, rule in zip(late["market"], late["day"], late["rule"])]
    named = late.dropna(subset=["rule's answer"])
    truth = named["up"].map({True: "up", False: "down"})
    return {"answers": len(named), "said up %": 100 * (named["answer"] == "up").mean(),
            "used 'reverse the 1-day move' %": 100 * (named["rule"] == "reverse the 1-day move").mean(),
            "answer agrees with its own rule %": 100 * (named["answer"] == named["rule's answer"]).mean(),
            "accuracy %": 100 * named["right"].mean(),
            "accuracy if it had obeyed its rule %": 100 * (named["rule's answer"] == truth).mean()}

pd.DataFrame({(version, d, k): late_check(runs[d, k], d) for version, runs in {"with lines": new, "test 3": old}.items()
              for d, k in RUNS}).T.round(1)

## Main results

- **Planted tester against its twin**, on the same days (McNemar). Test 3: +1.2 points.
- **Each new run against its test 3 version**, on the same days.

In [ ]:
def compare(a, b):
    both = a[["right"]].join(b[["right"]].add_suffix(" other"), how="inner")
    only_a, only_b, p = mcnemar(pd.Series(True, index=both.index), both["right"], both["right other"])
    return {"days": len(both), "accuracy %": 100 * both["right"].mean(), "other %": 100 * both["right other"].mean(),
            "difference, points": 100 * (both["right"].mean() - both["right other"].mean()), "p": p}

pd.DataFrame({
    "planted tester vs its twin (with lines)": compare(new["planted", "tester"], new["planted", "none"]),
    "planted tester vs its twin (test 3)": compare(old["planted", "tester"], old["planted", "none"]),
    **{f"{d} {k}: with lines vs test 3": compare(new[d, k], old[d, k]) for d, k in RUNS},
}).T.round(3)

## Any skill on real data?

AUC of the real none run, with the lines and without (test 3). The step 9 models with set A, on the same days, are below.

In [ ]:
pd.Series({"with lines": roc_auc_score(new["real", "none"]["up"], new["real", "none"]["p_up"]),
           "test 3": roc_auc_score(old["real", "none"]["up"], old["real", "none"]["p_up"])}, name="AUC").round(3).to_frame()

In [ ]:
import os
os.chdir("..")  # step 9's saved runs are found from the project folder
from more_inputs import MODELS, SAVED as STEP9
a = new["real", "none"].reset_index()
a["day"] = pd.to_datetime(a["day"])
rows = {}
for model in MODELS:
    ml = pd.concat([pd.read_pickle(STEP9 / f"A_close_to_close_{model}_{m}.pkl".replace(" ", "_")).assign(market=m)
                    for m in a["market"].unique()]).reset_index(names="day")
    both = a.merge(ml, on=["market", "day"], suffixes=("", " ml"))
    rows[model] = {"days": len(both), "AUC": roc_auc_score(both["up ml"], both["p_up ml"])}
os.chdir("llm_harness")
pd.DataFrame(rows).T.round(3)

CONCLUSIONS